# StockSmart — Per-Category DQN Inventory Optimization

This notebook implements the reinforcement learning pipeline:

1. **Custom Gymnasium Environments** — single-product `InventoryEnv` and multi-product `CategoryInventoryEnv`
2. **Per-Category DQN Training** — one shared DQN agent per product category, learning a generalized ordering policy
3. **Optional GA Pretraining** (DEAP) to seed the replay buffer with (s, S) policy trajectories
4. **Baseline Comparisons** against (s, S), EOQ, and random policies
5. **Evaluation & Visualization** across 5 product categories

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
import warnings

warnings.filterwarnings("ignore")

## 1. Configuration

In [ ]:
USE_GA_PRETRAINING = False
TOTAL_TIMESTEPS = 20_000
N_CATEGORIES = 5
EPISODE_LENGTH = 365

if USE_GA_PRETRAINING:
    from deap import base, creator, tools, algorithms

## 2. Load Artifacts from Data Processing Pipeline

In [ ]:
hourly_df = pd.read_parquet("artifacts/hourly_features.parquet")

try:
    forecast_df = pd.read_parquet("artifacts/rl_forecast_features.parquet")
    print(f"Forecast features loaded: {forecast_df.shape}")
except FileNotFoundError:
    forecast_df = None
    print("No forecast features found — proceeding without forecasts.")

with open("artifacts/feature_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

print(f"Hourly features: {hourly_df.shape}")
print(f"Unique series: {hourly_df['unique_id'].nunique()}")
print(f"Categories (first_category_id): {hourly_df['first_category_id'].nunique()}")

## 3. Single-Product Inventory Environment

Simulates daily inventory for one product-store. Kept for backward compatibility and per-product evaluation.

In [ ]:
class InventoryEnv(gym.Env):
    """
    Single product-store inventory management environment.

    State vector:
        [on_hand_inventory, *incoming_shipments(lead_time),
         *demand_forecast(horizon), price, discount, *stockout_history(N)]

    Action: discrete order quantity in {0, 1, 2, ..., max_order}
    Reward: -[holding_cost + stockout_penalty + ordering_cost]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        demand_series: np.ndarray,
        forecast_series: np.ndarray = None,
        covariate_matrix: np.ndarray = None,
        max_order: int = 50,
        lead_time: int = 2,
        initial_inventory: int = 20,
        holding_cost: float = 0.5,
        stockout_penalty: float = 5.0,
        fixed_order_cost: float = 25.0,
        variable_order_cost: float = 1.0,
        episode_length: int = 365,
        forecast_horizon: int = 7,
        stockout_history_len: int = 14,
    ):
        super().__init__()
        self.demand = demand_series
        self.forecasts = forecast_series
        self.covariates = covariate_matrix
        self.max_order = max_order
        self.lead_time = lead_time
        self.initial_inventory = initial_inventory
        self.holding_cost = holding_cost
        self.stockout_penalty = stockout_penalty
        self.fixed_order_cost = fixed_order_cost
        self.variable_order_cost = variable_order_cost
        self.episode_length = min(episode_length, len(demand_series))
        self.forecast_horizon = forecast_horizon
        self.stockout_history_len = stockout_history_len

        self.n_cov = covariate_matrix.shape[1] if covariate_matrix is not None else 0
        state_dim = (
            1 + lead_time + forecast_horizon + 2
            + self.n_cov + stockout_history_len
        )

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(state_dim,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(max_order + 1)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        self.inventory = self.initial_inventory
        self.pipeline = np.zeros(self.lead_time, dtype=np.float32)
        self.stockout_history = np.zeros(self.stockout_history_len, dtype=np.float32)
        self.total_cost = 0.0
        self.history = {"inventory": [], "demand": [], "order": [], "cost": [], "stockout": []}
        return self._get_obs(), {}

    def _get_obs(self):
        fc = np.zeros(self.forecast_horizon, dtype=np.float32)
        if self.forecasts is not None:
            end = min(self.t + self.forecast_horizon, len(self.forecasts))
            fc[:end - self.t] = self.forecasts[self.t:end]
        else:
            end = min(self.t + self.forecast_horizon, len(self.demand))
            fc[:end - self.t] = self.demand[self.t:end]

        cov = np.zeros(self.n_cov, dtype=np.float32)
        if self.covariates is not None and self.t < len(self.covariates):
            cov = self.covariates[self.t].astype(np.float32)

        return np.concatenate([
            [self.inventory], self.pipeline, fc,
            [1.0, 1.0], cov, self.stockout_history,
        ]).astype(np.float32)

    def step(self, action):
        order_qty = int(action)

        received = self.pipeline[0]
        self.pipeline = np.roll(self.pipeline, -1)
        self.pipeline[-1] = 0
        self.inventory += received

        if order_qty > 0:
            self.pipeline[-1] += order_qty

        demand = self.demand[self.t] if self.t < len(self.demand) else 0.0
        lost_sales = max(0, demand - self.inventory)
        self.inventory = max(0, self.inventory - demand)
        stockout = 1.0 if lost_sales > 0 else 0.0

        hold = self.holding_cost * max(self.inventory, 0)
        penalty = self.stockout_penalty * lost_sales
        order_cost = (self.fixed_order_cost * (order_qty > 0)
                      + self.variable_order_cost * order_qty)
        step_cost = hold + penalty + order_cost
        self.total_cost += step_cost

        self.stockout_history = np.roll(self.stockout_history, -1)
        self.stockout_history[-1] = stockout

        self.history["inventory"].append(self.inventory)
        self.history["demand"].append(demand)
        self.history["order"].append(order_qty)
        self.history["cost"].append(step_cost)
        self.history["stockout"].append(stockout)

        self.t += 1
        terminated = self.t >= self.episode_length
        reward = -step_cost

        return self._get_obs(), reward, terminated, False, {
            "step_cost": step_cost, "lost_sales": lost_sales, "stockout": stockout,
        }

    def render(self):
        print(f"t={self.t} | inv={self.inventory:.0f} | "
              f"pipeline={self.pipeline} | total_cost={self.total_cost:.1f}")

## 4. Multi-Product Category Inventory Environment

A shared environment for all product-stores in a category. On each `reset()` it randomly samples one product-store from the category, so the DQN learns a **generalized** ordering policy across the entire category.

In [ ]:
class CategoryInventoryEnv(gym.Env):
    """
    Multi-product inventory environment for a single product category.

    Holds demand series for ALL product-stores in the category.
    Each episode randomly selects one product-store to simulate,
    forcing the agent to learn a general policy.

    State vector:
        [product_index_normalized, on_hand_inventory, *incoming_shipments,
         *demand_forecast, price, discount, *stockout_history]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        demand_series_list: list,
        forecast_series_list: list = None,
        uid_list: list = None,
        max_order: int = 50,
        lead_time: int = 2,
        initial_inventory: int = 20,
        holding_cost: float = 0.5,
        stockout_penalty: float = 5.0,
        fixed_order_cost: float = 25.0,
        variable_order_cost: float = 1.0,
        episode_length: int = 365,
        forecast_horizon: int = 7,
        stockout_history_len: int = 14,
    ):
        super().__init__()
        self.demand_list = demand_series_list
        self.forecast_list = forecast_series_list
        self.uid_list = uid_list or [str(i) for i in range(len(demand_series_list))]
        self.n_products = len(demand_series_list)
        self.max_order = max_order
        self.lead_time = lead_time
        self.initial_inventory = initial_inventory
        self.holding_cost = holding_cost
        self.stockout_penalty = stockout_penalty
        self.fixed_order_cost = fixed_order_cost
        self.variable_order_cost = variable_order_cost
        self.episode_length = episode_length
        self.forecast_horizon = forecast_horizon
        self.stockout_history_len = stockout_history_len

        state_dim = (
            1                           # product index (normalized)
            + 1                         # on-hand inventory
            + lead_time                 # incoming shipments pipeline
            + forecast_horizon          # demand forecasts
            + 2                         # price, discount
            + stockout_history_len      # recent stockout indicators
        )

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(state_dim,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(max_order + 1)

        self._current_idx = 0
        self.demand = self.demand_list[0]
        self.forecasts = self.forecast_list[0] if self.forecast_list else None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._current_idx = self.np_random.integers(0, self.n_products)
        self.demand = self.demand_list[self._current_idx]
        self.forecasts = (
            self.forecast_list[self._current_idx]
            if self.forecast_list else None
        )
        self._ep_len = min(self.episode_length, len(self.demand))

        self.t = 0
        self.inventory = self.initial_inventory
        self.pipeline = np.zeros(self.lead_time, dtype=np.float32)
        self.stockout_history = np.zeros(self.stockout_history_len, dtype=np.float32)
        self.total_cost = 0.0
        self.history = {
            "inventory": [], "demand": [], "order": [],
            "cost": [], "stockout": [],
        }
        return self._get_obs(), {}

    def _get_obs(self):
        fc = np.zeros(self.forecast_horizon, dtype=np.float32)
        src = self.forecasts if self.forecasts is not None else self.demand
        end = min(self.t + self.forecast_horizon, len(src))
        fc[:end - self.t] = src[self.t:end]

        product_norm = self._current_idx / max(self.n_products - 1, 1)

        return np.concatenate([
            [product_norm],
            [self.inventory],
            self.pipeline,
            fc,
            [1.0, 1.0],
            self.stockout_history,
        ]).astype(np.float32)

    def step(self, action):
        order_qty = int(action)

        received = self.pipeline[0]
        self.pipeline = np.roll(self.pipeline, -1)
        self.pipeline[-1] = 0
        self.inventory += received

        if order_qty > 0:
            self.pipeline[-1] += order_qty

        demand = self.demand[self.t] if self.t < len(self.demand) else 0.0
        lost_sales = max(0, demand - self.inventory)
        self.inventory = max(0, self.inventory - demand)
        stockout = 1.0 if lost_sales > 0 else 0.0

        hold = self.holding_cost * max(self.inventory, 0)
        penalty = self.stockout_penalty * lost_sales
        order_cost = (self.fixed_order_cost * (order_qty > 0)
                      + self.variable_order_cost * order_qty)
        step_cost = hold + penalty + order_cost
        self.total_cost += step_cost

        self.stockout_history = np.roll(self.stockout_history, -1)
        self.stockout_history[-1] = stockout

        self.history["inventory"].append(self.inventory)
        self.history["demand"].append(demand)
        self.history["order"].append(order_qty)
        self.history["cost"].append(step_cost)
        self.history["stockout"].append(stockout)

        self.t += 1
        terminated = self.t >= self._ep_len
        reward = -step_cost

        return self._get_obs(), reward, terminated, False, {
            "step_cost": step_cost, "lost_sales": lost_sales, "stockout": stockout,
        }

    def current_uid(self):
        return self.uid_list[self._current_idx]

    def render(self):
        print(f"product={self.current_uid()} t={self.t} | "
              f"inv={self.inventory:.0f} | cost={self.total_cost:.1f}")

## 5. Environment Factories & Sanity Check

In [ ]:
def _get_daily_demand(uid, split):
    """Aggregate hourly demand to daily for a single product-store."""
    series = hourly_df[
        (hourly_df["unique_id"] == uid) & (hourly_df["split"] == split)
    ].sort_values("ds").copy()
    if len(series) == 0:
        return np.array([], dtype=np.float32)
    series["date"] = series["ds"].dt.date
    daily = series.groupby("date").agg({"y": "sum"}).reset_index()
    return daily["y"].values.astype(np.float32)


def make_env(uid, split="train", episode_length=365):
    """Create a single-product InventoryEnv for one unique_id."""
    demand = _get_daily_demand(uid, split)
    if len(demand) == 0:
        raise ValueError(f"No data for uid={uid}, split={split}")
    return InventoryEnv(
        demand_series=demand,
        max_order=50, lead_time=2, initial_inventory=20,
        episode_length=min(episode_length, len(demand)),
    )


def make_category_env(category_id, split="train", episode_length=365):
    """Create a CategoryInventoryEnv for all product-stores in a category."""
    cat_mask = hourly_df["first_category_id"] == category_id
    uids = hourly_df.loc[cat_mask, "unique_id"].unique().tolist()

    demand_list, uid_list = [], []
    for uid in uids:
        d = _get_daily_demand(uid, split)
        if len(d) >= 30:
            demand_list.append(d)
            uid_list.append(uid)

    if len(demand_list) == 0:
        raise ValueError(f"No valid series for category {category_id}, split={split}")

    return CategoryInventoryEnv(
        demand_series_list=demand_list,
        uid_list=uid_list,
        max_order=50, lead_time=2, initial_inventory=20,
        episode_length=episode_length,
    )


# Sanity check
sample_uid = hourly_df["unique_id"].unique()[0]
env = make_env(sample_uid, split="train")
obs, _ = env.reset()
print(f"Single-product env — Obs shape: {obs.shape}, Actions: {env.action_space}")

sample_cat = hourly_df["first_category_id"].value_counts().index[0]
cat_env = make_category_env(sample_cat, split="train")
obs, _ = cat_env.reset()
print(f"Category env (cat={sample_cat}) — Obs shape: {obs.shape}, "
      f"Products: {cat_env.n_products}, Actions: {cat_env.action_space}")

## 6. Select Top-5 Categories for Training

In [ ]:
cat_counts = (
    hourly_df.groupby("first_category_id")["unique_id"]
    .nunique()
    .sort_values(ascending=False)
)
top_categories = cat_counts.head(N_CATEGORIES).index.tolist()

print(f"Top {N_CATEGORIES} categories by number of product-store series:")
for cat_id in top_categories:
    print(f"  category {cat_id}: {cat_counts[cat_id]} product-store combos")

## 7. Baseline Policies

In [ ]:
def evaluate_policy(env, policy_fn, n_episodes=5):
    """Run a policy over multiple episodes and return average metrics."""
    total_costs, service_levels = [], []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action = policy_fn(obs, env)
            obs, _, done, _, _ = env.step(action)
        total_costs.append(env.total_cost)
        service_levels.append(1 - np.mean(env.history["stockout"]))
    return {
        "avg_cost": np.mean(total_costs),
        "std_cost": np.std(total_costs),
        "avg_service_level": np.mean(service_levels),
    }


def sS_policy(obs, env, s=10, S=30):
    """(s, S) policy: if inventory position <= s, order up to S."""
    inv_idx = 1 if isinstance(env, CategoryInventoryEnv) else 0
    inventory = obs[inv_idx]
    pipeline_total = obs[inv_idx + 1 : inv_idx + 1 + env.lead_time].sum()
    ip = inventory + pipeline_total
    if ip <= s:
        return min(int(S - ip), env.max_order)
    return 0


def eoq_policy(obs, env, Q=15, r=8):
    """EOQ policy: order Q when inventory position drops to r."""
    inv_idx = 1 if isinstance(env, CategoryInventoryEnv) else 0
    inventory = obs[inv_idx]
    pipeline_total = obs[inv_idx + 1 : inv_idx + 1 + env.lead_time].sum()
    if inventory + pipeline_total <= r:
        return min(Q, env.max_order)
    return 0


def run_episode(env, policy_fn):
    """Run a single episode and return the history dict."""
    obs, _ = env.reset()
    done = False
    while not done:
        action = policy_fn(obs, env)
        obs, _, done, _, _ = env.step(action)
    return env.history

## 8. Optional GA Pretraining (DEAP)

When `USE_GA_PRETRAINING = True`, evolves (s, S) policy parameters via genetic algorithm, then seeds the DQN replay buffer with GA trajectories. **Skipped by default** — the DQN trains from scratch with epsilon-greedy exploration.

In [ ]:
def run_ga_for_category(category_id, n_gen=20, pop_size=30):
    """Run GA to find best (s, S) for a category. Returns (best_s, best_S)."""
    if "FitnessMin" not in dir(creator):
        creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        creator.create("Individual", list, fitness=creator.FitnessMin)

    tb = base.Toolbox()
    tb.register("attr_s", np.random.randint, 1, 30)
    tb.register("attr_S", np.random.randint, 20, 60)
    tb.register("individual", tools.initCycle, creator.Individual,
                (tb.attr_s, tb.attr_S), n=1)
    tb.register("population", tools.initRepeat, list, tb.individual)

    def fitness(individual):
        s_val, S_val = int(individual[0]), int(individual[1])
        if s_val >= S_val:
            return (1e6,)
        env = make_category_env(category_id, split="train")
        costs = []
        for _ in range(3):
            obs, _ = env.reset()
            done = False
            while not done:
                action = sS_policy(obs, env, s=s_val, S=S_val)
                obs, _, done, _, _ = env.step(action)
            costs.append(env.total_cost)
        return (np.mean(costs),)

    tb.register("evaluate", fitness)
    tb.register("mate", tools.cxBlend, alpha=0.5)
    tb.register("mutate", tools.mutGaussian, mu=0, sigma=3, indpb=0.3)
    tb.register("select", tools.selTournament, tournsize=3)

    pop = tb.population(n=pop_size)
    hof = tools.HallOfFame(1)
    algorithms.eaSimple(pop, tb, cxpb=0.6, mutpb=0.3, ngen=n_gen,
                        halloffame=hof, verbose=False)

    best_s, best_S = int(hof[0][0]), int(hof[0][1])
    print(f"  GA best for category {category_id}: s={best_s}, S={best_S}")
    return best_s, best_S


def seed_replay_buffer(model, env, policy_fn, n_episodes=10):
    """Pre-fill the DQN replay buffer with trajectories from a policy."""
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action = policy_fn(obs, env)
            next_obs, reward, done, truncated, info = env.step(action)
            model.replay_buffer.add(
                obs=np.array([obs]),
                next_obs=np.array([next_obs]),
                action=np.array([action]),
                reward=np.array([reward]),
                done=np.array([done]),
                infos=[info],
            )
            obs = next_obs


if USE_GA_PRETRAINING:
    print("GA pretraining enabled.")
else:
    print("GA pretraining disabled — DQN trains from scratch.")

## 9. Train Per-Category DQN Agents

Loop over the top 5 categories. For each, build a `CategoryInventoryEnv`, create a DQN, optionally seed with GA trajectories, and train.

In [ ]:
category_models = {}
category_ga_params = {}

for cat_id in top_categories:
    print(f"\n{'='*60}")
    print(f"Category {cat_id}")
    print(f"{'='*60}")

    train_env_raw = make_category_env(cat_id, split="train", episode_length=EPISODE_LENGTH)
    print(f"  Products in category: {train_env_raw.n_products}")

    train_env = DummyVecEnv([lambda c=cat_id: make_category_env(c, "train", EPISODE_LENGTH)])
    val_env = DummyVecEnv([lambda c=cat_id: make_category_env(c, "val", EPISODE_LENGTH)])

    dqn = DQN(
        "MlpPolicy",
        train_env,
        learning_rate=1e-3,
        buffer_size=50_000,
        learning_starts=500,
        batch_size=64,
        gamma=0.99,
        target_update_interval=250,
        exploration_fraction=0.3,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.01,
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=0,
        seed=42,
    )

    if USE_GA_PRETRAINING:
        best_s, best_S = run_ga_for_category(cat_id)
        category_ga_params[cat_id] = (best_s, best_S)
        ga_fn = lambda obs, env, s=best_s, S=best_S: sS_policy(obs, env, s=s, S=S)
        seed_replay_buffer(dqn, train_env_raw, ga_fn, n_episodes=10)
        print(f"  Replay buffer seeded: {dqn.replay_buffer.size()} transitions")

    save_dir = f"./artifacts/dqn_category_{cat_id}"
    os.makedirs(save_dir, exist_ok=True)

    eval_cb = EvalCallback(
        val_env,
        best_model_save_path=save_dir,
        log_path=save_dir,
        eval_freq=2000,
        n_eval_episodes=3,
        deterministic=True,
        verbose=0,
    )

    print(f"  Training DQN for {TOTAL_TIMESTEPS} timesteps...")
    dqn.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_cb, progress_bar=True)
    model_path = os.path.join(save_dir, "dqn_final")
    dqn.save(model_path)
    category_models[cat_id] = model_path
    print(f"  Saved to {model_path}")

print(f"\nTrained {len(category_models)} per-category DQN agents.")

## 10. Evaluation — Per-Category DQN vs Baselines

Compare each category's DQN agent against (s, S), EOQ, and random baselines on the test split.

In [ ]:
all_results = []

for cat_id in top_categories:
    print(f"\nEvaluating category {cat_id}...")
    test_env = make_category_env(cat_id, split="test", episode_length=EPISODE_LENGTH)

    loaded_dqn = DQN.load(category_models[cat_id])
    def dqn_fn(obs, env, _m=loaded_dqn):
        action, _ = _m.predict(obs, deterministic=True)
        return int(action)

    for policy_name, pfn in [
        ("Random", lambda o, e: e.action_space.sample()),
        ("(s,S)", lambda o, e: sS_policy(o, e, s=10, S=30)),
        ("EOQ", lambda o, e: eoq_policy(o, e, Q=15, r=8)),
        ("DQN", dqn_fn),
    ]:
        res = evaluate_policy(test_env, pfn, n_episodes=5)
        all_results.append({
            "category_id": cat_id,
            "n_products": test_env.n_products,
            "policy": policy_name,
            **res,
        })

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 70)
print("TEST SET RESULTS — Per-Category DQN vs Baselines")
print("=" * 70)
pivot = results_df.pivot_table(
    index=["category_id", "n_products"],
    columns="policy",
    values=["avg_cost", "avg_service_level"],
)
print(pivot.round(2))

## 11. Inventory Trajectory Visualization

For each category, plot inventory levels and cumulative cost for one sample product: DQN vs (s, S) baseline.

In [ ]:
n_cats = len(top_categories)
fig, axes = plt.subplots(n_cats, 2, figsize=(18, 4 * n_cats), squeeze=False)

for row, cat_id in enumerate(top_categories):
    test_env = make_category_env(cat_id, split="test", episode_length=EPISODE_LENGTH)

    loaded_dqn = DQN.load(category_models[cat_id])
    def dqn_fn(obs, env, _m=loaded_dqn):
        action, _ = _m.predict(obs, deterministic=True)
        return int(action)

    policies = {
        "DQN": dqn_fn,
        "(s,S)": lambda o, e: sS_policy(o, e, s=10, S=30),
    }

    for name, pfn in policies.items():
        hist = run_episode(test_env, pfn)
        t = range(len(hist["inventory"]))
        axes[row, 0].plot(t, hist["inventory"], label=name, alpha=0.8)
        axes[row, 1].plot(t, np.cumsum(hist["cost"]), label=name, alpha=0.8)

    axes[row, 0].plot(t, hist["demand"], label="Demand", color="black",
                       linewidth=0.7, linestyle="--", alpha=0.5)
    axes[row, 0].set_ylabel("Inventory / Demand")
    axes[row, 0].set_title(f"Category {cat_id} — Inventory")
    axes[row, 0].legend(fontsize=8)

    axes[row, 1].set_ylabel("Cumulative Cost")
    axes[row, 1].set_title(f"Category {cat_id} — Cost")
    axes[row, 1].legend(fontsize=8)

axes[-1, 0].set_xlabel("Day")
axes[-1, 1].set_xlabel("Day")
plt.tight_layout()
plt.show()

## 12. W&B Experiment Logging (Optional)

In [ ]:
# Uncomment to enable W&B logging
# import wandb
#
# wandb.init(project="stocksmart-rl", config={
#     "n_categories": N_CATEGORIES,
#     "use_ga": USE_GA_PRETRAINING,
#     "dqn_timesteps": TOTAL_TIMESTEPS,
#     "dqn_arch": [256, 256],
#     "categories": top_categories,
# })
#
# for _, row in results_df.iterrows():
#     wandb.log({
#         f"cat_{row['category_id']}/{row['policy']}/avg_cost": row["avg_cost"],
#         f"cat_{row['category_id']}/{row['policy']}/service_level": row["avg_service_level"],
#     })
#
# wandb.finish()

print("W&B logging cell ready — uncomment and run after `wandb login`.")

## Next Steps

- **Scale up**: Increase `TOTAL_TIMESTEPS` to 100k+ and `SAMPLE_N_SERIES` in data processing
- **PPO alternative**: Swap `DQN` for `PPO` in Stable-Baselines3 for continuous action spaces
- **Hyperparameter sweeps**: Use W&B sweeps over learning rate, network size, discount factor
- **Ablation studies**: Enable `USE_GA_PRETRAINING = True` and compare against pure DQN
- **Full hierarchy**: Extend per-category to per-subcategory or per-cluster grouping